# Create Sample Dataset for Fast Iteration

This notebook creates a smaller sample of the training data for faster experimentation.
We'll sample complete sequences to maintain data integrity and create both train/validation splits.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

np.random.seed(42)

## 1. Load Full Dataset

In [2]:
# Load full training data
data_path = '../competition_package/datasets/train.parquet'
df_full = pd.read_parquet(data_path)

print(f"Full dataset: {df_full.shape[0]:,} rows × {df_full.shape[1]} columns")
print(f"Number of sequences: {df_full['seq_ix'].nunique()}")
print(f"Memory: {df_full.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Full dataset: 517,000 rows × 35 columns
Number of sequences: 517
Memory: 138.05 MB


## 2. Create Sample (50 sequences for dev, ~10% of data)

In [3]:
# Sample complete sequences
all_sequences = df_full['seq_ix'].unique()
n_sample_sequences = 50

sample_seq_ids = np.random.choice(all_sequences, size=n_sample_sequences, replace=False)
df_sample = df_full[df_full['seq_ix'].isin(sample_seq_ids)].copy()

print(f"Sample dataset: {df_sample.shape[0]:,} rows × {df_sample.shape[1]} columns")
print(f"Number of sequences: {df_sample['seq_ix'].nunique()}")
print(f"Memory: {df_sample.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Reduction: {100 * (1 - len(df_sample) / len(df_full)):.1f}%")

Sample dataset: 50,000 rows × 35 columns
Number of sequences: 50
Memory: 13.73 MB
Reduction: 90.3%


## 3. Create Train/Val Split (80/20 by sequences)

In [4]:
# Split sample sequences into train/val
n_train = int(0.8 * n_sample_sequences)
train_seq_ids = sample_seq_ids[:n_train]
val_seq_ids = sample_seq_ids[n_train:]

df_train = df_sample[df_sample['seq_ix'].isin(train_seq_ids)].copy()
df_val = df_sample[df_sample['seq_ix'].isin(val_seq_ids)].copy()

print("Train set:")
print(f"  Sequences: {df_train['seq_ix'].nunique()}")
print(f"  Rows: {len(df_train):,}")
print(f"  Memory: {df_train.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\nValidation set:")
print(f"  Sequences: {df_val['seq_ix'].nunique()}")
print(f"  Rows: {len(df_val):,}")
print(f"  Memory: {df_val.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Train set:
  Sequences: 40
  Rows: 40,000
  Memory: 10.99 MB

Validation set:
  Sequences: 10
  Rows: 10,000
  Memory: 2.75 MB


## 4. Save Sample Datasets

In [5]:
# Create data directory
Path('data').mkdir(exist_ok=True)

# Save sample datasets
df_sample.to_parquet('data/sample_full.parquet', index=False)
df_train.to_parquet('data/sample_train.parquet', index=False)
df_val.to_parquet('data/sample_val.parquet', index=False)

print("Sample datasets saved:")
print("  - data/sample_full.parquet (all 50 sequences)")
print("  - data/sample_train.parquet (40 sequences for training)")
print("  - data/sample_val.parquet (10 sequences for validation)")

Sample datasets saved:
  - data/sample_full.parquet (all 50 sequences)
  - data/sample_train.parquet (40 sequences for training)
  - data/sample_val.parquet (10 sequences for validation)


## 5. Verify Sample Quality

In [6]:
# Check that sequences are complete
seq_lengths = df_sample.groupby('seq_ix').size()
print(f"All sequences have 1000 steps: {(seq_lengths == 1000).all()}")
print(f"Sequence lengths: {seq_lengths.unique()}")

# Check train/val overlap
print(f"\nNo sequence overlap between train/val: {len(set(train_seq_ids) & set(val_seq_ids)) == 0}")

# Check feature distributions
feature_cols = [col for col in df_sample.columns if col not in ['seq_ix', 'step_in_seq', 'need_prediction']]
print(f"\nNumber of features: {len(feature_cols)}")

# Compare sample vs full dataset statistics
print("\nFeature mean comparison (first 5 features):")
for feat in feature_cols[:5]:
    full_mean = df_full[feat].mean()
    sample_mean = df_sample[feat].mean()
    diff_pct = 100 * abs(full_mean - sample_mean) / abs(full_mean) if full_mean != 0 else 0
    print(f"  {feat}: Full={full_mean:.6f}, Sample={sample_mean:.6f}, Diff={diff_pct:.2f}%")

All sequences have 1000 steps: True
Sequence lengths: [1000]

No sequence overlap between train/val: True

Number of features: 32

Feature mean comparison (first 5 features):
  0: Full=0.001247, Sample=0.030016, Diff=2306.96%
  1: Full=0.022680, Sample=0.020160, Diff=11.11%
  2: Full=-0.006079, Sample=0.013771, Diff=326.53%
  3: Full=-0.013578, Sample=-0.008514, Diff=37.29%
  4: Full=-0.001519, Sample=0.002384, Diff=256.97%


## 6. Summary

**Sample Dataset Created:**
- 50 sequences (10% of full 517 sequences)
- Train: 40 sequences
- Validation: 10 sequences
- Each sequence: 1,000 steps
- Features: 32

**Use for:**
- Rapid prototyping
- Hyperparameter tuning
- Model architecture experiments

**Final evaluation:**
- Train on full dataset
- Test on hidden test set